# register-back-fn-after-wrap — worked example 2: Register the Backward Function for Negation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For a custom negation operation `neg(x) = -x`, the gradient with respect to `x` is simply `-grad_out`. When building a framework with a registration-based autograd dispatcher, you define a back function and register it at `(fwd_fn, argnum=0)`. The dispatcher then retrieves and calls it without needing to know the specific operation.

## Worked solution

**Step 1 — math check.** If `out = -x`, then `d(out)/d(x) = -1`. By the chain rule, `grad_x = grad_out * (-1) = -grad_out`.

**Step 2 — define `neg_back`.** Signature `(grad_out, out, x) -> Tensor`. We only need `grad_out` here: return `-grad_out`.

**Step 3 — define a custom forward function.** We use `lambda x: -x` to represent negation. Store it in a variable so it has a stable identity for use as a dict key.

**Step 4 — register.** Call `BACK_FUNCS.add_back_func(neg_fn, 0, neg_back)`. Since negation is unary, only argnum=0 is needed.

**Step 5 — dispatch and verify.** Retrieve with `get_back_func(neg_fn, 0)` and call with a test gradient. Expect `grad_x = -grad_out`.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def neg_fn(x: t.Tensor) -> t.Tensor:
    return -x

def neg_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    # d(-x)/dx = -1, so grad flows back negated
    return -grad_out

def register_neg(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(neg_fn, 0, neg_back)

# --- exercise and print ---
BACK_FUNCS = BackwardFuncLookup()
register_neg(BACK_FUNCS)

x = t.tensor([2.0, -3.0, 0.5])
out = neg_fn(x)
grad_out = t.tensor([1.0, 1.0, 1.0])

back_fn = BACK_FUNCS.get_back_func(neg_fn, 0)
grad_x = back_fn(grad_out, out, x)
print('grad_out:', grad_out)
print('grad_x:  ', grad_x)  # Should be [-1, -1, -1]
print('match:', t.allclose(grad_x, -grad_out))